# MultiStream — reinferencia completa

Ejecuta la siguiente celda en Kaggle. El notebook es autónomo: carga los datos y folds, reconstruye MultiStream, procesa las 10 seeds × 5 folds, guarda resultados por ventana y sujeto, y genera el heatmap y los archivos `.dat` para TikZ.

In [1]:

# ============================================================
# MULTISTREAM — REINFERENCIA COMPLETA
# 10 seeds × 5 folds
#
# Este archivo es autónomo:
#   1) carga folds.pkl;
#   2) carga y segmenta los 120 sujetos;
#   3) construye los tres streams del modelo;
#   4) reconstruye la arquitectura MultiStream original;
#   5) carga los 50 mejores checkpoints;
#   6) repite la inferencia por ventanas;
#   7) calcula resultados por fold, seed y sujeto;
#   8) genera el heatmap y los archivos .dat para TikZ.
# ============================================================

from __future__ import annotations

import gc
import json
import math
import pickle
import re
import warnings
from collections import Counter
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.signal import welch
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

MODEL_ROOT = Path(
    "/kaggle/input/datasets/alejandragomezr/models-cte-net/"
    "resultados_multistream_tdah_ARTICULO-20260715T223803Z-1-001/"
    "resultados_multistream_tdah_ARTICULO"
)

FOLDS_PATH = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH/folds.pkl"
)

ADHD_DIR = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH/ieee/ADHD_group"
)

CONTROL_DIR = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH/ieee/Control_group"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/MultiStream_reinference_complete"
)

FIGURE_DIR = OUTPUT_ROOT / "figures"
TIKZ_DIR = FIGURE_DIR / "tikz_data"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TIKZ_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "MultiStream"
SEEDS = list(range(10))
FOLDS_TO_RUN = list(range(5))  # índices 0,1,2,3,4 de folds.pkl

WINDOW_SIZE = 512
OVERLAP = 0.50
FS = 128
N_TEMP_WINDOWS = 10
EXPECTED_CHANNELS = 19

BATCH_SIZE = 256
NUM_WORKERS = 0
METRIC_TOLERANCE = 1e-8
STRICT_METRIC_MATCH = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 2. UTILIDADES GENERALES
# ============================================================

def normalize_subject_id(value: Any) -> str:
    """Normaliza rutas, bytes o nombres de archivo a un subject_id."""
    if isinstance(value, bytes):
        value = value.decode("utf-8")

    name = Path(str(value)).name

    for suffix in (".mat", ".npy", ".npz", ".csv"):
        if name.lower().endswith(suffix):
            name = name[:-len(suffix)]
            break

    return name.strip()


def normalize_folds(folds_raw: Any) -> List[Tuple[List[str], List[str], List[str]]]:
    """
    Convierte folds.pkl al formato:
        [(train_subjects, val_subjects, test_subjects), ...]
    """

    if isinstance(folds_raw, dict):
        if "folds" in folds_raw:
            fold_items = folds_raw["folds"]
        else:
            fold_items = [
                folds_raw[key]
                for key in sorted(folds_raw.keys(), key=str)
            ]
    else:
        fold_items = folds_raw

    normalized: List[Tuple[List[str], List[str], List[str]]] = []

    for fold_idx, fold_item in enumerate(fold_items):
        if isinstance(fold_item, (list, tuple)) and len(fold_item) == 3:
            train_subjects, val_subjects, test_subjects = fold_item

        elif isinstance(fold_item, dict):
            def find_first(keys: Sequence[str]) -> Any:
                for key in keys:
                    if key in fold_item:
                        return fold_item[key]
                return None

            train_subjects = find_first(
                ["train", "train_subjects", "subjects_train",
                 "train_idx", "train_indices", "idx_train"]
            )
            val_subjects = find_first(
                ["val", "validation", "valid", "val_subjects",
                 "subjects_val", "val_idx", "validation_idx", "idx_val"]
            )
            test_subjects = find_first(
                ["test", "test_subjects", "subjects_test",
                 "test_idx", "test_indices", "idx_test"]
            )

            if train_subjects is None or val_subjects is None or test_subjects is None:
                raise ValueError(
                    f"No se pudieron identificar train/val/test en fold {fold_idx}. "
                    f"Claves: {list(fold_item.keys())}"
                )
        else:
            raise TypeError(
                f"Formato de fold no reconocido en posición {fold_idx}: "
                f"{type(fold_item).__name__}"
            )

        train = [normalize_subject_id(s) for s in train_subjects]
        val = [normalize_subject_id(s) for s in val_subjects]
        test = [normalize_subject_id(s) for s in test_subjects]

        normalized.append((train, val, test))

    return normalized


def build_mat_index(directory: Path) -> Dict[str, Path]:
    """Construye subject_id -> archivo .mat."""
    if not directory.exists():
        raise FileNotFoundError(f"No existe la carpeta:\n{directory}")

    index = {
        normalize_subject_id(path.stem): path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".mat"
    }

    if not index:
        raise FileNotFoundError(f"No se encontraron .mat en:\n{directory}")

    return index


def load_original_eeg_mat(
    mat_path: Path,
    expected_channels: int = EXPECTED_CHANNELS,
) -> Tuple[np.ndarray, str]:
    """
    Replica el cargador original:
        data = scipy.io.loadmat(path)
        columna = list(data.keys())[-1]
        eeg = data[columna].T

    Devuelve (canales, tiempo).
    """
    data = scipy.io.loadmat(mat_path)
    variable_name = list(data.keys())[-1]
    eeg = np.asarray(data[variable_name]).T.astype(np.float32)

    if eeg.ndim != 2:
        raise ValueError(
            f"{mat_path.name}: la variable {variable_name!r} no es 2-D; "
            f"shape={eeg.shape}"
        )

    if eeg.shape[0] != expected_channels:
        raise ValueError(
            f"{mat_path.name}: se esperaban {expected_channels} canales "
            f"después de transponer, pero shape={eeg.shape}"
        )

    if not np.isfinite(eeg).all():
        raise ValueError(f"{mat_path.name} contiene NaN o Inf.")

    return eeg, variable_name


def calculate_binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
) -> Dict[str, float]:
    """Métricas utilizadas en el entrenamiento original."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    try:
        auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        auc = float("nan")

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                average="binary",
                zero_division=0,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                average="binary",
                zero_division=0,
            )
        ),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "auc": auc,
    }


def summarize_values(values: Iterable[float]) -> Dict[str, float]:
    arr = np.asarray(list(values), dtype=float)
    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": (
            float(np.nanstd(arr, ddof=1))
            if len(arr) > 1 else 0.0
        ),
        "n": int(len(arr)),
    }


# ============================================================
# 3. CARGA Y SEGMENTACIÓN DE LOS 120 SUJETOS
# ============================================================

def process_adhd_subjects(
    folds_path: Path = FOLDS_PATH,
    adhd_dir: Path = ADHD_DIR,
    control_dir: Path = CONTROL_DIR,
    window_size: int = WINDOW_SIZE,
    overlap: float = OVERLAP,
    expected_channels: int = EXPECTED_CHANNELS,
) -> Dict[str, Any]:
    """
    Carga únicamente los sujetos contenidos en folds.pkl.

    Conserva el orden del código original:
        controles ordenados alfabéticamente;
        después TDAH ordenados alfabéticamente.

    Genera ventanas de 512 muestras con 50 % de solapamiento.
    """

    if not 0 <= overlap < 1:
        raise ValueError("overlap debe pertenecer a [0,1).")

    stride = int(round(window_size * (1.0 - overlap)))
    if stride <= 0:
        raise ValueError("El stride debe ser positivo.")

    with open(folds_path, "rb") as file:
        folds = normalize_folds(pickle.load(file))

    if len(folds) != 5:
        raise ValueError(
            f"Se esperaban 5 folds; se encontraron {len(folds)}."
        )

    protocol_subjects: set[str] = set()
    test_counter: Counter = Counter()

    for fold_idx, (train, val, test) in enumerate(folds):
        train_set, val_set, test_set = set(train), set(val), set(test)

        if train_set & val_set:
            raise ValueError(f"Solapamiento train-val en fold {fold_idx}.")
        if train_set & test_set:
            raise ValueError(f"Solapamiento train-test en fold {fold_idx}.")
        if val_set & test_set:
            raise ValueError(f"Solapamiento val-test en fold {fold_idx}.")

        protocol_subjects.update(train_set | val_set | test_set)
        test_counter.update(test)

    invalid_test_counts = {
        subject: count
        for subject, count in test_counter.items()
        if count != 1
    }

    if invalid_test_counts:
        raise ValueError(
            "Cada sujeto debe aparecer exactamente una vez en test: "
            f"{invalid_test_counts}"
        )

    adhd_files = build_mat_index(adhd_dir)
    control_files = build_mat_index(control_dir)

    overlap_ids = set(adhd_files) & set(control_files)
    if overlap_ids:
        raise ValueError(
            f"Identificadores presentes en ambas clases: {sorted(overlap_ids)}"
        )

    available_subjects = set(adhd_files) | set(control_files)
    missing = protocol_subjects - available_subjects
    if missing:
        raise FileNotFoundError(
            f"Sujetos de folds.pkl sin archivo .mat: {sorted(missing)}"
        )

    unused_subjects = sorted(available_subjects - protocol_subjects)

    control_subjects = sorted(protocol_subjects & set(control_files))
    adhd_subjects = sorted(protocol_subjects & set(adhd_files))

    if len(control_subjects) != 60 or len(adhd_subjects) != 60:
        raise ValueError(
            "La cohorte no es 60/60: "
            f"Control={len(control_subjects)}, ADHD={len(adhd_subjects)}"
        )

    subject_records = [
        {
            "subject_id": subject,
            "label": 0,
            "class_name": "Control",
            "path": control_files[subject],
        }
        for subject in control_subjects
    ] + [
        {
            "subject_id": subject,
            "label": 1,
            "class_name": "ADHD",
            "path": adhd_files[subject],
        }
        for subject in adhd_subjects
    ]

    windows: List[np.ndarray] = []
    labels: List[int] = []
    subject_ids: List[str] = []
    window_ids: List[int] = []
    metadata_rows: List[Dict[str, Any]] = []

    global_window_index = 0

    for subject_position, record in enumerate(subject_records):
        subject_id = record["subject_id"]
        label = int(record["label"])
        class_name = record["class_name"]
        mat_path = record["path"]

        eeg, variable_name = load_original_eeg_mat(
            mat_path,
            expected_channels=expected_channels,
        )

        _, n_samples = eeg.shape
        if n_samples < window_size:
            raise ValueError(
                f"{subject_id}: {n_samples} muestras < {window_size}."
            )

        subject_window_id = 0

        for start_sample in range(
            0,
            n_samples - window_size + 1,
            stride,
        ):
            end_sample = start_sample + window_size
            window = eeg[:, start_sample:end_sample]

            windows.append(window)
            labels.append(label)
            subject_ids.append(subject_id)
            window_ids.append(subject_window_id)

            metadata_rows.append({
                "global_window_index": global_window_index,
                "subject_position": subject_position,
                "subject_id": subject_id,
                "label": label,
                "class_name": class_name,
                "window_id": subject_window_id,
                "window_name": f"Window {subject_window_id + 1}",
                "start_sample": start_sample,
                "end_sample": end_sample,
                "n_recording_samples": n_samples,
                "mat_variable": variable_name,
                "mat_filename": mat_path.name,
            })

            subject_window_id += 1
            global_window_index += 1

    X = np.stack(windows).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    subject_ids_array = np.asarray(subject_ids, dtype=str)
    window_ids_array = np.asarray(window_ids, dtype=np.int64)
    metadata = pd.DataFrame(metadata_rows)

    if set(subject_ids_array) != protocol_subjects:
        raise RuntimeError("No se procesaron todos los sujetos del protocolo.")

    print("=" * 80)
    print("DATOS PROCESADOS")
    print("=" * 80)
    print("Device:", DEVICE)
    print("X:", X.shape, X.dtype)
    print("y:", y.shape, y.dtype)
    print("Sujetos:", len(np.unique(subject_ids_array)))
    print("Control:", len(control_subjects))
    print("ADHD:", len(adhd_subjects))
    print("No utilizado:", unused_subjects)
    print("Window size:", window_size)
    print("Stride:", stride)

    return {
        "X": X,
        "y": y,
        "subject_ids": subject_ids_array,
        "window_ids": window_ids_array,
        "metadata": metadata,
        "folds": folds,
        "protocol_subjects": sorted(protocol_subjects),
        "unused_subjects": unused_subjects,
        "window_size": window_size,
        "stride": stride,
    }


# ============================================================
# 4. CONSTRUCCIÓN DE LOS TRES STREAMS
# ============================================================

def prepare_streams_4s(
    X: np.ndarray,
    fs: int = FS,
    n_win: int = N_TEMP_WINDOWS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Función original de MultiStream.

    X:
        (N, 19, 512)

    Returns:
        freq: (N, 20, 1)
        temp: (N, 10, 1)
        spat: (N, 19, 1)
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"X debe ser 3-D; shape={X.shape}")

    N, C, T = X.shape

    if T != 512:
        raise ValueError(
            "MultiStream espera exactamente 512 muestras por ventana."
        )

    # 1) Stream espectral: 20 bandas logarítmicas.
    log_bands = np.logspace(
        np.log10(1),
        np.log10(fs / 2),
        21,
    )
    bands = list(zip(log_bands[:-1], log_bands[1:]))

    def band_power(signal: np.ndarray) -> np.ndarray:
        frequencies, pxx = welch(
            signal,
            fs=fs,
            nperseg=512,
        )

        return np.asarray([
            pxx[(frequencies >= low) & (frequencies < high)].mean()
            if np.any(
                (frequencies >= low) & (frequencies < high)
            )
            else 0.0
            for low, high in bands
        ])

    print("\nConstruyendo stream espectral...")
    freq_features = np.stack([
        [band_power(channel) for channel in trial]
        for trial in X
    ])

    freq = freq_features.mean(axis=1)[..., None]

    # 2) Stream temporal: 10 ventanas.
    temporal_window_size = T // n_win
    usable_samples = n_win * temporal_window_size

    x_trim = X[:, :, :usable_samples]
    temporal_windows = x_trim.reshape(
        N,
        C,
        n_win,
        temporal_window_size,
    )

    temp = temporal_windows.mean(axis=(1, 3))[..., None]

    # 3) Stream espacial: RMS por canal.
    spat = np.sqrt((X ** 2).mean(axis=2))[..., None]

    freq = freq.astype(np.float32)
    temp = temp.astype(np.float32)
    spat = spat.astype(np.float32)

    print("freq:", freq.shape, freq.dtype)
    print("temp:", temp.shape, temp.dtype)
    print("spat:", spat.shape, spat.dtype)

    return freq, temp, spat


# ============================================================
# 5. ARQUITECTURA MULTISTREAM ORIGINAL
# ============================================================

class PositionalEncoding(nn.Module):
    """Codificación posicional sinusoidal fija del código original."""

    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        self.d_model = d_model

        pos = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        feature_index = torch.arange(
            d_model,
            dtype=torch.float32,
        ).unsqueeze(0)

        exponent = (
            2.0 * torch.floor(feature_index / 2.0)
        ) / float(d_model)

        angle_rates = 1.0 / torch.pow(
            torch.tensor(10000.0),
            exponent,
        )

        angle_rads = pos * angle_rates

        sin = torch.sin(angle_rads[:, 0::2])
        cos = torch.cos(angle_rads[:, 1::2])

        positional_encoding = torch.cat(
            [sin, cos],
            dim=-1,
        ).unsqueeze(0)

        self.register_buffer(
            "pos_encoding",
            positional_encoding,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence_length = x.size(1)

        return (
            x.float()
            + self.pos_encoding[
                :, :sequence_length, :
            ].to(x.device)
        )


class KerasStyleMultiHeadSelfAttention(nn.Module):
    """
    Replica tf.keras.layers.MultiHeadAttention con:
        num_heads=num_heads
        key_dim=d_model
    """

    def __init__(
        self,
        d_model: int = 64,
        num_heads: int = 4,
        key_dim: Optional[int] = None,
    ):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.key_dim = (
            key_dim if key_dim is not None else d_model
        )
        self.inner_dim = self.num_heads * self.key_dim

        self.q_proj = nn.Linear(d_model, self.inner_dim)
        self.k_proj = nn.Linear(d_model, self.inner_dim)
        self.v_proj = nn.Linear(d_model, self.inner_dim)
        self.out_proj = nn.Linear(self.inner_dim, d_model)

        self.scale = self.key_dim ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.key_dim,
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.key_dim,
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.key_dim,
        ).transpose(1, 2)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        ) * self.scale

        attention = torch.softmax(scores, dim=-1)
        context = torch.matmul(attention, v)

        context = context.transpose(1, 2).contiguous()
        context = context.view(
            batch_size,
            sequence_length,
            self.inner_dim,
        )

        return self.out_proj(context)


class TransformerEncoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int = 64,
        num_heads: int = 4,
        d_ff: int = 128,
        rate: float = 0.1,
    ):
        super().__init__()

        self.attn = KerasStyleMultiHeadSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            key_dim=d_model,
        )

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout1 = nn.Dropout(rate)
        self.dropout2 = nn.Dropout(rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attention_output = self.attn(x)
        attention_output = self.dropout1(attention_output)

        out1 = self.layernorm1(x + attention_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        return self.layernorm2(out1 + ffn_output)


class StreamEncoder(nn.Module):
    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        d_model: int = 64,
        num_layers: int = 2,
        num_heads: int = 4,
        d_ff: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.seq_len = seq_len
        self.feat_dim = feat_dim

        self.projection = nn.Linear(feat_dim, d_model)
        self.pos_encoding = PositionalEncoding(d_model=d_model)

        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                rate=dropout,
            )
            for _ in range(num_layers)
        ])

        self.decoder = nn.Linear(d_model, 128)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.projection(x)
        x = self.pos_encoding(x)

        for block in self.transformer_blocks:
            x = block(x)

        x = x.mean(dim=1)
        x = F.relu(self.decoder(x))

        return x


class EEGAttentionTransformer(nn.Module):
    """Modelo MultiStream base del notebook original."""

    def __init__(
        self,
        freq_shape: Tuple[int, int] = (20, 1),
        temp_shape: Tuple[int, int] = (10, 1),
        spat_shape: Tuple[int, int] = (19, 1),
        d_model: int = 64,
        num_layers: int = 2,
        num_heads: int = 4,
        d_ff: int = 128,
        num_classes: int = 2,
    ):
        super().__init__()

        self.freq_stream = StreamEncoder(
            seq_len=freq_shape[0],
            feat_dim=freq_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.temp_stream = StreamEncoder(
            seq_len=temp_shape[0],
            feat_dim=temp_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.spat_stream = StreamEncoder(
            seq_len=spat_shape[0],
            feat_dim=spat_shape[1],
            d_model=d_model,
            num_layers=num_layers,
            num_heads=num_heads,
            d_ff=d_ff,
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(
        self,
        freq: torch.Tensor,
        temp: torch.Tensor,
        spat: torch.Tensor,
        return_proba: bool = False,
    ) -> torch.Tensor:
        freq_vector = self.freq_stream(freq)
        temp_vector = self.temp_stream(temp)
        spat_vector = self.spat_stream(spat)

        features = torch.cat(
            [freq_vector, temp_vector, spat_vector],
            dim=1,
        )

        logits = self.classifier(features)

        if return_proba:
            return torch.softmax(logits, dim=1)

        return logits


class MultiStreamReturnDictWrapper(nn.Module):
    """Wrapper usado durante el entrenamiento original."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(
        self,
        freq: torch.Tensor,
        temp: torch.Tensor,
        spat: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        logits = self.model(freq, temp, spat)

        return {
            "logits": logits,
            "out_activation": torch.softmax(logits, dim=1),
        }


def build_multistream_model() -> MultiStreamReturnDictWrapper:
    base_model = EEGAttentionTransformer(
        freq_shape=(20, 1),
        temp_shape=(10, 1),
        spat_shape=(19, 1),
        d_model=64,
        num_layers=2,
        num_heads=4,
        d_ff=128,
        num_classes=2,
    )

    return MultiStreamReturnDictWrapper(base_model)


# ============================================================
# 6. CHECKPOINTS Y MÉTRICAS GUARDADAS
# ============================================================

def safe_torch_load(path: Path, map_location: torch.device) -> Any:
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=True,
        )
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )


def extract_state_dict(loaded_object: Any) -> Dict[str, torch.Tensor]:
    if not isinstance(loaded_object, dict):
        raise TypeError(
            "El checkpoint no contiene un diccionario."
        )

    for key in (
        "model_state_dict",
        "state_dict",
        "model",
        "weights",
    ):
        if key in loaded_object and isinstance(loaded_object[key], dict):
            state_dict = loaded_object[key]
            break
    else:
        state_dict = loaded_object

    clean: Dict[str, torch.Tensor] = {}

    for key, value in state_dict.items():
        clean_key = str(key)

        while clean_key.startswith("module."):
            clean_key = clean_key[len("module."):]

        clean[clean_key] = value

    return clean


def load_state_dict_robust(
    model: nn.Module,
    checkpoint_path: Path,
    device: torch.device,
) -> None:
    loaded = safe_torch_load(checkpoint_path, device)
    state_dict = extract_state_dict(loaded)

    attempts: List[Dict[str, torch.Tensor]] = [state_dict]

    # Checkpoint de modelo base -> agregar "model."
    if state_dict and not all(
        key.startswith("model.") for key in state_dict
    ):
        attempts.append({
            f"model.{key}": value
            for key, value in state_dict.items()
        })

    # Checkpoint del wrapper -> quitar "model."
    if state_dict and all(
        key.startswith("model.") for key in state_dict
    ):
        attempts.append({
            key[len("model."):]: value
            for key, value in state_dict.items()
        })

    errors: List[str] = []

    for candidate in attempts:
        try:
            model.load_state_dict(candidate, strict=True)
            return
        except RuntimeError as error:
            errors.append(str(error))

    raise RuntimeError(
        f"No fue posible cargar:\n{checkpoint_path}\n\n"
        + "\n\n".join(errors)
    )


def find_seed_directory(
    model_root: Path,
    seed: int,
) -> Path:
    exact_candidates = [
        model_root / f"MultiStream_TDAH_fixed_seed_{seed}",
        model_root / f"MultiStream_TDAH_fixed_seed{seed}",
        model_root / f"multistream_TDAH_fixed_seed_{seed}",
        model_root / f"multistream_tdah_fixed_seed_{seed}",
    ]

    for candidate in exact_candidates:
        if candidate.is_dir():
            return candidate

    pattern = re.compile(
        rf"seed[_-]?{seed}$",
        flags=re.IGNORECASE,
    )

    matches = sorted(
        path
        for path in model_root.rglob("*")
        if path.is_dir() and pattern.search(path.name)
    )

    if len(matches) == 1:
        return matches[0]

    if not matches:
        raise FileNotFoundError(
            f"No se encontró la carpeta de seed={seed} dentro de:\n"
            f"{model_root}"
        )

    raise RuntimeError(
        f"Se encontraron varias carpetas para seed={seed}:\n"
        + "\n".join(str(path) for path in matches)
    )


def find_fold_directory(
    seed_directory: Path,
    fold_index: int,
) -> Path:
    """
    Detecta de forma inequívoca una de estas convenciones:
        fold_01 ... fold_05
        fold_1  ... fold_5
        fold_00 ... fold_04
        fold_0  ... fold_4
    """
    conventions = [
        [f"fold_{number:02d}" for number in range(1, 6)],
        [f"fold_{number}" for number in range(1, 6)],
        [f"fold_{number:02d}" for number in range(0, 5)],
        [f"fold_{number}" for number in range(0, 5)],
    ]

    for names in conventions:
        paths = [seed_directory / name for name in names]

        if all(path.is_dir() for path in paths):
            return paths[fold_index]

    matches = sorted(
        path
        for path in seed_directory.iterdir()
        if path.is_dir() and path.name.lower().startswith("fold_")
    )

    if len(matches) == 5:
        def fold_number(path: Path) -> int:
            match = re.search(r"(\d+)$", path.name)
            return int(match.group(1)) if match else 10**9

        matches = sorted(matches, key=fold_number)
        return matches[fold_index]

    raise FileNotFoundError(
        f"No se encontró el fold índice {fold_index} en:\n"
        f"{seed_directory}\n"
        f"Directorios disponibles: {[p.name for p in matches]}"
    )


def find_checkpoint(fold_directory: Path) -> Path:
    candidate_names = [
        "best_model_weights.pt",
        "best_state.pt",
        "best_model.pt",
        "model_best.pt",
    ]

    for filename in candidate_names:
        candidate = fold_directory / filename
        if candidate.is_file():
            return candidate

    recursive_matches: List[Path] = []

    for pattern in (
        "*best*weight*.pt",
        "*best*state*.pt",
        "*best*model*.pt",
    ):
        recursive_matches.extend(
            fold_directory.rglob(pattern)
        )

    recursive_matches = sorted(set(recursive_matches))

    if len(recursive_matches) == 1:
        return recursive_matches[0]

    raise FileNotFoundError(
        f"No se encontró un checkpoint de mejor modelo en:\n"
        f"{fold_directory}\n"
        f"Archivos .pt: {[p.name for p in fold_directory.rglob('*.pt')]}"
    )


def metrics_from_json(path: Path) -> Optional[Dict[str, float]]:
    try:
        with open(path, "r", encoding="utf-8") as file:
            obj = json.load(file)
    except Exception:
        return None

    candidates = [
        obj.get("test_metrics"),
        obj.get("fold_metrics"),
        obj.get("metrics"),
        obj,
    ]

    required = {
        "accuracy",
        "balanced_accuracy",
        "recall",
        "precision",
        "kappa",
        "auc",
    }

    for candidate in candidates:
        if isinstance(candidate, dict) and "accuracy" in candidate:
            result: Dict[str, float] = {}

            for key in required:
                value = candidate.get(key, np.nan)
                try:
                    result[key] = float(value)
                except (TypeError, ValueError):
                    result[key] = float("nan")

            return result

    return None


def metrics_from_pickle(path: Path) -> Optional[Dict[str, float]]:
    try:
        with open(path, "rb") as file:
            obj = pickle.load(file)
    except Exception:
        return None

    if not isinstance(obj, dict):
        return None

    candidates = [
        obj.get("test_metrics"),
        obj.get("fold_metrics"),
        obj.get("metrics"),
        obj,
    ]

    required = {
        "accuracy",
        "balanced_accuracy",
        "recall",
        "precision",
        "kappa",
        "auc",
    }

    for candidate in candidates:
        if isinstance(candidate, dict) and "accuracy" in candidate:
            result: Dict[str, float] = {}

            for key in required:
                value = candidate.get(key, np.nan)
                try:
                    result[key] = float(value)
                except (TypeError, ValueError):
                    result[key] = float("nan")

            return result

    return None


def find_saved_fold_metrics(
    model_root: Path,
    fold_directory: Path,
    seed: int,
    fold_index: int,
) -> Optional[Dict[str, float]]:
    """Busca primero fold_results.json y luego otras variantes."""
    for filename in (
        "fold_results.json",
        "fold_result.json",
        "fold_metrics.json",
    ):
        path = fold_directory / filename

        if path.is_file():
            metrics = metrics_from_json(path)
            if metrics is not None:
                return metrics

    for filename in (
        "fold_result.pkl",
        "fold_results.pkl",
    ):
        path = fold_directory / filename

        if path.is_file():
            metrics = metrics_from_pickle(path)
            if metrics is not None:
                return metrics

    # Respaldo: repeated_test_results.csv.
    repeated_csv_candidates = [
        model_root / "repeated_test_results.csv",
        model_root.parent / "repeated_test_results.csv",
    ]

    repeated_csv_candidates.extend(
        model_root.rglob("repeated_test_results.csv")
    )

    for csv_path in repeated_csv_candidates:
        if not csv_path.is_file():
            continue

        try:
            frame = pd.read_csv(csv_path)
        except Exception:
            continue

        required_columns = {"seed", "fold", "accuracy"}

        if not required_columns.issubset(frame.columns):
            continue

        fold_values = sorted(
            frame["fold"].dropna().astype(int).unique().tolist()
        )

        if fold_values == [1, 2, 3, 4, 5]:
            target_fold = fold_index + 1
        elif fold_values == [0, 1, 2, 3, 4]:
            target_fold = fold_index
        else:
            continue

        row = frame[
            (frame["seed"].astype(int) == int(seed))
            & (frame["fold"].astype(int) == int(target_fold))
        ]

        if len(row) == 1:
            record = row.iloc[0]
            return {
                key: float(record[key])
                if key in record and pd.notna(record[key])
                else float("nan")
                for key in (
                    "accuracy",
                    "balanced_accuracy",
                    "recall",
                    "precision",
                    "kappa",
                    "auc",
                )
            }

    return None


# ============================================================
# 7. INFERENCIA
# ============================================================

def get_fold_test_data(
    data: Dict[str, Any],
    freq: np.ndarray,
    temp: np.ndarray,
    spat: np.ndarray,
    fold_index: int,
) -> Dict[str, Any]:
    _, _, test_subjects = data["folds"][fold_index]

    test_mask = np.isin(
        data["subject_ids"],
        np.asarray(test_subjects, dtype=str),
    )

    metadata = (
        data["metadata"]
        .loc[test_mask]
        .reset_index(drop=True)
        .copy()
    )

    output = {
        "freq": freq[test_mask],
        "temp": temp[test_mask],
        "spat": spat[test_mask],
        "y_true": data["y"][test_mask],
        "subject_ids": data["subject_ids"][test_mask],
        "window_ids": data["window_ids"][test_mask],
        "metadata": metadata,
        "test_subjects": list(test_subjects),
    }

    n_subjects = len(np.unique(output["subject_ids"]))

    if n_subjects != 24:
        raise RuntimeError(
            f"Fold {fold_index}: se esperaban 24 sujetos; "
            f"se encontraron {n_subjects}."
        )

    lengths = {
        len(output["freq"]),
        len(output["temp"]),
        len(output["spat"]),
        len(output["y_true"]),
        len(output["subject_ids"]),
        len(output["metadata"]),
    }

    if len(lengths) != 1:
        raise RuntimeError(
            f"Fold {fold_index}: longitudes incompatibles."
        )

    return output


def run_multistream_inference(
    model: nn.Module,
    freq: np.ndarray,
    temp: np.ndarray,
    spat: np.ndarray,
    batch_size: int = BATCH_SIZE,
    device: torch.device = DEVICE,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(freq, dtype=np.float32)
        ),
        torch.from_numpy(
            np.asarray(temp, dtype=np.float32)
        ),
        torch.from_numpy(
            np.asarray(spat, dtype=np.float32)
        ),
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    model.eval()
    probability_batches: List[np.ndarray] = []

    with torch.inference_mode():
        for freq_batch, temp_batch, spat_batch in loader:
            freq_batch = freq_batch.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )
            temp_batch = temp_batch.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )
            spat_batch = spat_batch.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            outputs = model(
                freq_batch,
                temp_batch,
                spat_batch,
            )

            if isinstance(outputs, dict):
                probabilities = outputs["out_activation"]
            else:
                probabilities = torch.softmax(outputs, dim=1)

            probability_batches.append(
                probabilities.detach().cpu().numpy()
            )

    probabilities_2c = np.concatenate(
        probability_batches,
        axis=0,
    ).astype(np.float32)

    y_prob_control = probabilities_2c[:, 0]
    y_prob_adhd = probabilities_2c[:, 1]
    y_pred = np.argmax(
        probabilities_2c,
        axis=1,
    ).astype(np.int64)

    return y_prob_control, y_prob_adhd, y_pred


# ============================================================
# 8. RESÚMENES Y EXPORTACIÓN
# ============================================================

def build_subject_summary_by_seed(
    window_predictions: pd.DataFrame,
) -> pd.DataFrame:
    summary = (
        window_predictions
        .groupby(
            [
                "model",
                "seed",
                "fold",
                "subject_id",
                "label",
                "class_name",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            n_windows=("window_id", "size"),
            n_correct_windows=("correct", "sum"),
            window_accuracy=("correct", "mean"),
            mean_prob_adhd=("y_prob_adhd", "mean"),
            std_prob_adhd=("y_prob_adhd", "std"),
        )
    )

    summary["window_accuracy_percent"] = (
        100.0 * summary["window_accuracy"]
    )

    return summary


def build_subject_accuracy_across_seeds(
    subject_summary_by_seed: pd.DataFrame,
) -> pd.DataFrame:
    summary = (
        subject_summary_by_seed
        .groupby(
            [
                "model",
                "subject_id",
                "label",
                "class_name",
            ],
            as_index=False,
            sort=False,
        )
        .agg(
            fold=("fold", "first"),
            n_seeds=("seed", "nunique"),
            n_windows_per_seed=("n_windows", "first"),
            mean_window_accuracy=("window_accuracy", "mean"),
            std_window_accuracy=("window_accuracy", "std"),
            median_window_accuracy=("window_accuracy", "median"),
            min_window_accuracy=("window_accuracy", "min"),
            max_window_accuracy=("window_accuracy", "max"),
            mean_prob_adhd=("mean_prob_adhd", "mean"),
        )
    )

    summary["mean_window_accuracy_percent"] = (
        100.0 * summary["mean_window_accuracy"]
    )

    summary["std_window_accuracy_percent"] = (
        100.0 * summary["std_window_accuracy"]
    )

    invalid = summary[summary["n_seeds"] != len(SEEDS)]

    if not invalid.empty:
        raise RuntimeError(
            "Hay sujetos sin las diez seeds:\n"
            f"{invalid[['subject_id', 'n_seeds']]}"
        )

    return summary


def create_heatmap_and_tikz_data(
    subject_accuracy: pd.DataFrame,
    figure_dir: Path = FIGURE_DIR,
    tikz_dir: Path = TIKZ_DIR,
) -> pd.DataFrame:
    ordered = (
        subject_accuracy
        .sort_values(
            [
                "label",
                "mean_window_accuracy_percent",
                "subject_id",
            ],
            ascending=[True, True, True],
        )
        .reset_index(drop=True)
    )

    n_control = int((ordered["label"] == 0).sum())
    n_adhd = int((ordered["label"] == 1).sum())

    if n_control != 60 or n_adhd != 60:
        raise RuntimeError(
            f"Distribución inesperada: {n_control}/{n_adhd}"
        )

    ordered.insert(
        0,
        "heatmap_column",
        np.arange(1, len(ordered) + 1),
    )

    order_csv = figure_dir / (
        "MultiStream_heatmap_subject_order.csv"
    )

    ordered[
        [
            "heatmap_column",
            "subject_id",
            "label",
            "class_name",
            "fold",
            "mean_window_accuracy_percent",
            "std_window_accuracy_percent",
        ]
    ].to_csv(order_csv, index=False)

    values = (
        ordered["mean_window_accuracy_percent"]
        .to_numpy(dtype=float)
        .reshape(1, -1)
    )

    fig, ax = plt.subplots(figsize=(20, 3.2))

    image = ax.imshow(
        values,
        aspect="auto",
        interpolation="nearest",
        cmap="viridis",
        vmin=0,
        vmax=100,
    )

    ax.set_yticks([0])
    ax.set_yticklabels(
        ["MultiStream"],
        fontsize=11,
        fontweight="bold",
    )

    tick_positions = np.arange(
        0,
        len(ordered),
        5,
        dtype=int,
    )

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(
        ordered.iloc[tick_positions]["subject_id"],
        rotation=90,
        fontsize=7,
    )
    ax.set_xlim(-0.5, len(ordered) - 0.5)

    ax.axvline(
        n_control - 0.5,
        linewidth=1.5,
        linestyle="--",
    )

    ax.text(
        0.25,
        1.12,
        f"Control (n={n_control})",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

    ax.text(
        0.75,
        1.12,
        f"ADHD (n={n_adhd})",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

    ax.set_xlabel(
        "Subjects ordered from lower to higher mean window accuracy "
        "within each class",
        fontsize=10,
    )

    ax.set_title(
        "MultiStream subject-wise performance averaged across ten seeds",
        fontsize=13,
        pad=34,
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.025,
        pad=0.015,
    )
    colorbar.set_label(
        "Mean window accuracy (%)",
        fontsize=10,
    )

    fig.tight_layout()

    png_path = figure_dir / (
        "MultiStream_heatmap_mean_by_subject.png"
    )
    pdf_path = figure_dir / (
        "MultiStream_heatmap_mean_by_subject.pdf"
    )

    fig.savefig(
        png_path,
        dpi=400,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    # --------------------------------------------------------
    # DAT principal para TikZ
    # --------------------------------------------------------

    heatmap_dat = ordered.copy()
    heatmap_dat["x"] = np.arange(1, len(heatmap_dat) + 1)
    heatmap_dat["y"] = 0.0

    heatmap_dat = heatmap_dat[
        [
            "x",
            "y",
            "subject_id",
            "label",
            "class_name",
            "fold",
            "mean_window_accuracy_percent",
            "std_window_accuracy_percent",
        ]
    ].rename(
        columns={
            "mean_window_accuracy_percent": "accuracy",
            "std_window_accuracy_percent": "std_accuracy",
        }
    )

    heatmap_dat_path = (
        tikz_dir
        / "MultiStream_heatmap_mean_by_subject.dat"
    )

    heatmap_dat.to_csv(
        heatmap_dat_path,
        sep=" ",
        index=False,
        float_format="%.6f",
    )

    ticks_dat = pd.DataFrame({
        "x": tick_positions + 1,
        "subject_id": (
            ordered.iloc[tick_positions]["subject_id"]
            .astype(str)
            .tolist()
        ),
    })

    ticks_dat_path = (
        tikz_dir
        / "MultiStream_heatmap_subject_ticks.dat"
    )

    ticks_dat.to_csv(
        ticks_dat_path,
        sep=" ",
        index=False,
    )

    classes_dat = pd.DataFrame([
        {
            "class_name": "Control",
            "label": 0,
            "n_subjects": n_control,
            "x_start": 0.5,
            "x_end": n_control + 0.5,
            "x_center": (1 + n_control) / 2,
        },
        {
            "class_name": "ADHD",
            "label": 1,
            "n_subjects": n_adhd,
            "x_start": n_control + 0.5,
            "x_end": n_control + n_adhd + 0.5,
            "x_center": (
                n_control + (n_adhd + 1) / 2
            ),
        },
    ])

    classes_dat.to_csv(
        tikz_dir / "MultiStream_heatmap_classes.dat",
        sep=" ",
        index=False,
        float_format="%.1f",
    )

    latex_xticks = ",".join(
        ticks_dat["x"].astype(str)
    )
    latex_xticklabels = ",".join(
        ticks_dat["subject_id"].astype(str)
    )

    with open(
        tikz_dir / "MultiStream_heatmap_axis_settings.tex",
        "w",
        encoding="utf-8",
    ) as file:
        file.write("% Generado automáticamente\n")
        file.write(
            f"\\def\\MultiStreamXTicks{{{latex_xticks}}}\n"
        )
        file.write(
            "\\def\\MultiStreamXTickLabels"
            f"{{{latex_xticklabels}}}\n"
        )
        file.write(
            f"\\def\\MultiStreamControlN{{{n_control}}}\n"
        )
        file.write(
            f"\\def\\MultiStreamADHDN{{{n_adhd}}}\n"
        )
        file.write(
            "\\def\\MultiStreamClassDivider"
            f"{{{n_control + 0.5}}}\n"
        )
        file.write(
            "\\def\\MultiStreamControlCenter"
            f"{{{(1 + n_control) / 2:.1f}}}\n"
        )
        file.write(
            "\\def\\MultiStreamADHDCenter"
            f"{{{n_control + (n_adhd + 1) / 2:.1f}}}\n"
        )

    print("\nHeatmap guardado en:")
    print(png_path)
    print(pdf_path)
    print("\nDatos TikZ guardados en:")
    print(heatmap_dat_path)

    return ordered


# ============================================================
# 9. EJECUCIÓN COMPLETA
# ============================================================

def main() -> None:
    if not MODEL_ROOT.exists():
        raise FileNotFoundError(
            f"No existe MODEL_ROOT:\n{MODEL_ROOT}"
        )

    print("=" * 90)
    print("MULTISTREAM — REINFERENCIA COMPLETA")
    print("=" * 90)
    print("Device:", DEVICE)
    print("MODEL_ROOT:", MODEL_ROOT)
    print("OUTPUT_ROOT:", OUTPUT_ROOT)

    # 1) EEG segmentado.
    data = process_adhd_subjects()

    # 2) Streams. Se calculan una sola vez.
    freq, temp, spat = prepare_streams_4s(
        data["X"],
        fs=FS,
        n_win=N_TEMP_WINDOWS,
    )

    all_window_frames: List[pd.DataFrame] = []
    fold_metric_rows: List[Dict[str, Any]] = []
    comparison_rows: List[Dict[str, Any]] = []

    # 3) 10 seeds × 5 folds.
    for seed in SEEDS:
        seed_directory = find_seed_directory(
            MODEL_ROOT,
            seed,
        )

        for fold_index in FOLDS_TO_RUN:
            fold_directory = find_fold_directory(
                seed_directory,
                fold_index,
            )

            checkpoint_path = find_checkpoint(
                fold_directory
            )

            fold_data = get_fold_test_data(
                data,
                freq,
                temp,
                spat,
                fold_index,
            )

            model = build_multistream_model().to(DEVICE)

            load_state_dict_robust(
                model,
                checkpoint_path,
                DEVICE,
            )

            (
                y_prob_control,
                y_prob_adhd,
                y_pred,
            ) = run_multistream_inference(
                model=model,
                freq=fold_data["freq"],
                temp=fold_data["temp"],
                spat=fold_data["spat"],
                batch_size=BATCH_SIZE,
                device=DEVICE,
            )

            y_true = fold_data["y_true"]

            recalculated_metrics = calculate_binary_metrics(
                y_true=y_true,
                y_pred=y_pred,
                y_prob=y_prob_adhd,
            )

            saved_metrics = find_saved_fold_metrics(
                model_root=MODEL_ROOT,
                fold_directory=fold_directory,
                seed=seed,
                fold_index=fold_index,
            )

            saved_accuracy = (
                saved_metrics["accuracy"]
                if saved_metrics is not None
                else np.nan
            )

            recalculated_accuracy = (
                recalculated_metrics["accuracy"]
            )

            accuracy_difference = (
                recalculated_accuracy - saved_accuracy
                if np.isfinite(saved_accuracy)
                else np.nan
            )

            accuracy_matches = (
                bool(
                    np.isclose(
                        recalculated_accuracy,
                        saved_accuracy,
                        atol=METRIC_TOLERANCE,
                        rtol=0.0,
                    )
                )
                if np.isfinite(saved_accuracy)
                else None
            )

            print(
                f"seed={seed} fold={fold_index} | "
                f"guardado={saved_accuracy:.10f} | "
                f"recalculado={recalculated_accuracy:.10f}"
            )

            metric_row: Dict[str, Any] = {
                "model": MODEL_NAME,
                "seed": seed,
                "fold": fold_index,
                "original_fold_number": fold_index + 1,
                "n_test_windows": len(y_true),
                "n_test_subjects": len(
                    np.unique(fold_data["subject_ids"])
                ),
                "checkpoint_path": str(checkpoint_path),
                **recalculated_metrics,
            }

            fold_metric_rows.append(metric_row)

            comparison_row: Dict[str, Any] = {
                "model": MODEL_NAME,
                "seed": seed,
                "fold": fold_index,
                "original_fold_number": fold_index + 1,
                "saved_accuracy": saved_accuracy,
                "recomputed_accuracy": recalculated_accuracy,
                "accuracy_difference": accuracy_difference,
                "accuracy_matches": accuracy_matches,
            }

            for metric_name in (
                "balanced_accuracy",
                "recall",
                "precision",
                "kappa",
                "auc",
            ):
                saved_value = (
                    saved_metrics.get(metric_name, np.nan)
                    if saved_metrics is not None
                    else np.nan
                )
                recomputed_value = recalculated_metrics[
                    metric_name
                ]

                comparison_row[
                    f"saved_{metric_name}"
                ] = saved_value
                comparison_row[
                    f"recomputed_{metric_name}"
                ] = recomputed_value
                comparison_row[
                    f"{metric_name}_difference"
                ] = (
                    recomputed_value - saved_value
                    if np.isfinite(saved_value)
                    else np.nan
                )

            comparison_rows.append(comparison_row)

            predictions = fold_data["metadata"].copy()

            predictions.insert(0, "model", MODEL_NAME)
            predictions.insert(1, "seed", seed)
            predictions.insert(2, "fold", fold_index)

            predictions["y_true"] = y_true.astype(
                np.int64
            )
            predictions["y_prob_control"] = (
                y_prob_control.astype(np.float32)
            )
            predictions["y_prob_adhd"] = (
                y_prob_adhd.astype(np.float32)
            )
            predictions["y_pred"] = y_pred.astype(
                np.int64
            )
            predictions["correct"] = (
                predictions["y_pred"]
                == predictions["y_true"]
            ).astype(np.int64)

            all_window_frames.append(predictions)

            del model
            gc.collect()

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 4) Consolidar predicciones y métricas.
    # --------------------------------------------------------

    window_predictions = pd.concat(
        all_window_frames,
        ignore_index=True,
    )

    fold_metrics = pd.DataFrame(fold_metric_rows)
    comparison = pd.DataFrame(comparison_rows)

    window_predictions_path = (
        OUTPUT_ROOT
        / "MultiStream_all_window_predictions.csv"
    )
    fold_metrics_path = (
        OUTPUT_ROOT
        / "MultiStream_fold_metrics_recomputed.csv"
    )
    comparison_path = (
        OUTPUT_ROOT
        / "MultiStream_saved_vs_recomputed_metrics.csv"
    )

    window_predictions.to_csv(
        window_predictions_path,
        index=False,
    )
    fold_metrics.to_csv(
        fold_metrics_path,
        index=False,
    )
    comparison.to_csv(
        comparison_path,
        index=False,
    )

    # --------------------------------------------------------
    # 5) Resumen por sujeto y seed.
    # --------------------------------------------------------

    subject_by_seed = build_subject_summary_by_seed(
        window_predictions
    )

    subject_across_seeds = (
        build_subject_accuracy_across_seeds(
            subject_by_seed
        )
    )

    subject_by_seed_path = (
        OUTPUT_ROOT
        / "MultiStream_subject_summary_by_seed.csv"
    )
    subject_across_seeds_path = (
        OUTPUT_ROOT
        / "MultiStream_subject_accuracy_across_seeds.csv"
    )

    subject_by_seed.to_csv(
        subject_by_seed_path,
        index=False,
    )
    subject_across_seeds.to_csv(
        subject_across_seeds_path,
        index=False,
    )

    # --------------------------------------------------------
    # 6) Resumen sobre 50 folds.
    # --------------------------------------------------------

    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "recall",
        "precision",
        "kappa",
        "auc",
    ]

    summary_50_rows = []

    for metric_name in metric_names:
        values = fold_metrics[metric_name].to_numpy(
            dtype=float
        )
        stats = summarize_values(values)

        summary_50_rows.append({
            "metric": metric_name,
            **stats,
        })

    summary_50 = pd.DataFrame(summary_50_rows)

    summary_50_path = (
        OUTPUT_ROOT
        / "MultiStream_summary_50_runs.csv"
    )

    summary_50.to_csv(
        summary_50_path,
        index=False,
    )

    # --------------------------------------------------------
    # 7) Promedio de cinco folds para cada seed.
    # --------------------------------------------------------

    seed_metrics = (
        fold_metrics
        .groupby("seed", as_index=False)[metric_names]
        .mean()
    )

    seed_metrics_path = (
        OUTPUT_ROOT
        / "MultiStream_seed_metrics_mean_of_folds.csv"
    )

    seed_metrics.to_csv(
        seed_metrics_path,
        index=False,
    )

    summary_10_rows = []

    for metric_name in metric_names:
        stats = summarize_values(
            seed_metrics[metric_name]
        )

        summary_10_rows.append({
            "metric": metric_name,
            **stats,
        })

    summary_10 = pd.DataFrame(summary_10_rows)

    summary_10_path = (
        OUTPUT_ROOT
        / "MultiStream_summary_across_10_seeds.csv"
    )

    summary_10.to_csv(
        summary_10_path,
        index=False,
    )

    # --------------------------------------------------------
    # 8) Validación de reproducción.
    # --------------------------------------------------------

    available_comparisons = comparison[
        comparison["accuracy_matches"].notna()
    ]

    all_match = (
        bool(
            available_comparisons[
                "accuracy_matches"
            ].astype(bool).all()
        )
        if len(available_comparisons) > 0
        else None
    )

    print("\nTodos coinciden:", all_match)
    print("\nResumen sobre los 50 folds:")
    print(
        summary_50[
            ["metric", "mean", "std_sample", "n"]
        ].to_string(index=False)
    )

    print("\nResumen sobre las 10 seeds:")
    print(
        summary_10[
            ["metric", "mean", "std_sample", "n"]
        ].to_string(index=False)
    )

    # --------------------------------------------------------
    # 9) Heatmap y TikZ.
    # --------------------------------------------------------

    ordered_subjects = create_heatmap_and_tikz_data(
        subject_across_seeds
    )

    # --------------------------------------------------------
    # 10) JSON final.
    # --------------------------------------------------------

    final_summary = {
        "model": MODEL_NAME,
        "device": str(DEVICE),
        "model_root": str(MODEL_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "n_subjects": int(
            subject_across_seeds["subject_id"].nunique()
        ),
        "n_seeds": int(len(SEEDS)),
        "n_folds": int(len(FOLDS_TO_RUN)),
        "n_test_runs": int(len(fold_metrics)),
        "all_saved_accuracies_match": all_match,
        "metric_tolerance": METRIC_TOLERANCE,
        "input_shapes": {
            "X": list(data["X"].shape),
            "freq": list(freq.shape),
            "temp": list(temp.shape),
            "spat": list(spat.shape),
        },
        "summary_over_50_folds": {
            row["metric"]: {
                "mean": float(row["mean"]),
                "std_sample": float(row["std_sample"]),
                "n": int(row["n"]),
            }
            for _, row in summary_50.iterrows()
        },
        "summary_across_10_seeds": {
            row["metric"]: {
                "mean": float(row["mean"]),
                "std_sample": float(row["std_sample"]),
                "n": int(row["n"]),
            }
            for _, row in summary_10.iterrows()
        },
        "files": {
            "window_predictions": str(
                window_predictions_path
            ),
            "subject_summary_by_seed": str(
                subject_by_seed_path
            ),
            "subject_accuracy_across_seeds": str(
                subject_across_seeds_path
            ),
            "fold_metrics": str(fold_metrics_path),
            "saved_vs_recomputed": str(comparison_path),
            "summary_50_runs": str(summary_50_path),
            "seed_metrics": str(seed_metrics_path),
            "summary_10_seeds": str(summary_10_path),
        },
    }

    summary_json_path = (
        OUTPUT_ROOT
        / "MultiStream_reinference_summary.json"
    )

    with open(
        summary_json_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            final_summary,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\n" + "=" * 90)
    print("PROCESO FINALIZADO")
    print("=" * 90)
    print("Archivos guardados en:", OUTPUT_ROOT)
    print("\nPrimeros sujetos:")
    print(
        subject_across_seeds.head().to_string(
            index=False
        )
    )

    if (
        STRICT_METRIC_MATCH
        and all_match is False
    ):
        mismatches = comparison[
            comparison["accuracy_matches"] == False
        ][
            [
                "seed",
                "fold",
                "saved_accuracy",
                "recomputed_accuracy",
                "accuracy_difference",
            ]
        ]

        raise RuntimeError(
            "La inferencia terminó y los archivos fueron guardados, "
            "pero algunas accuracies no coinciden:\n"
            f"{mismatches.to_string(index=False)}"
        )


if __name__ == "__main__":
    main()


MULTISTREAM — REINFERENCIA COMPLETA
Device: cpu
MODEL_ROOT: /kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_multistream_tdah_ARTICULO-20260715T223803Z-1-001/resultados_multistream_tdah_ARTICULO
OUTPUT_ROOT: /kaggle/working/MultiStream_reinference_complete
DATOS PROCESADOS
Device: cpu
X: (8213, 19, 512) float32
y: (8213,) int64
Sujetos: 120
Control: 60
ADHD: 60
No utilizado: ['v36p']
Window size: 512
Stride: 256

Construyendo stream espectral...
freq: (8213, 20, 1) float32
temp: (8213, 10, 1) float32
spat: (8213, 19, 1) float32


FileNotFoundError: No se encontró un checkpoint de mejor modelo en:
/kaggle/input/datasets/alejandragomezr/models-cte-net/resultados_multistream_tdah_ARTICULO-20260715T223803Z-1-001/resultados_multistream_tdah_ARTICULO/MultiStream_TDAH_fixed_seed_0/fold_01
Archivos .pt: []